# 02 -- Timestral zero-shot

Scores Timestral (5M and 209M, zero-shot, no fine-tuning) on **exactly** the same evaluation set as notebook 01: same tickers, same forecast origins (every 5 test-period days), same 30-day horizon, same `realized_var`. This notebook needs a GPU-enabled `torch` + the `timestral` package -- it will not run in the plain local venv used for `src/` and its tests.


In [ ]:
import os
import sys

try:
    import google.colab  # noqa: F401
    from google.colab import drive

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    DATA = "/content/drive/MyDrive/timestral-vol-data"
    os.makedirs(f"{DATA}/results", exist_ok=True)
    os.makedirs(f"{DATA}/results/ts_cache", exist_ok=True)
    os.environ["HF_HOME"] = f"{DATA}/hf_cache"

    REPO_DIR = "/content/timestral-vol"
    REPO_URL = "https://github.com/MachaGotJuice/timestral-vol.git"
    if os.path.isdir(REPO_DIR):
        os.chdir(REPO_DIR)
        !git pull
    else:
        !git clone {REPO_URL} {REPO_DIR}
        os.chdir(REPO_DIR)
    sys.path.insert(0, REPO_DIR)
    !pip install -q -r requirements.txt

    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        pass
else:
    DATA = None
    os.makedirs("results/ts_cache", exist_ok=True)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.data import (
    DEFAULT_START,
    DEFAULT_TICKERS,
    download_ohlc,
    forecast_origins,
    garman_klass,
    realized_forward,
    target_30d,
)
from src.metrics import dm_test, qlike
from src.timestral_utils import forecast_quantiles, load_model, nearest_quantile, quantile_mean

plt.rcParams["figure.figsize"] = (12, 4)

HORIZON = 30
STEP = 5
CONTEXT_LEN = 1024
TEST_START = "2022-01-01"
CACHE_DIR = f"{DATA}/results/ts_cache" if DATA else "results/ts_cache"
BASELINES_PATH = f"{DATA}/results/forecasts_baselines.csv" if DATA and os.path.exists(f"{DATA}/results/forecasts_baselines.csv") else "results/forecasts_baselines.csv"


## 1. Why we feed the 30-day average, and why we read step 30

Instead of giving Timestral the raw daily Garman-Klass variance, we feed it `y = target_30d(daily_var)`: the same trailing 30-day log-mean variance series notebook 01's HAR model is built on (`src.data.target_30d`). The payoff is an exact algebraic identity: `y[t+30] = log(realized_var at origin t)`. So forecasting `y` 30 steps ahead and reading the 30th step (index 29) **is** forecasting the realized target directly -- no extra aggregation needed on our end, and the model is already seeing a smoothed, less noisy input.

A quick check that this identity actually holds on real data, then a small figure showing it:


In [ ]:
example_df = download_ohlc("SPY", start=DEFAULT_START, cache_dir="cache")
example_daily_var = garman_klass(example_df)
example_y = target_30d(example_daily_var, window=30)

example_t = forecast_origins(example_daily_var.index, TEST_START, horizon=HORIZON, step=STEP)[0]
t_pos = example_daily_var.index.get_loc(example_t)
t_plus_h = example_daily_var.index[t_pos + HORIZON]

lhs = example_y.loc[t_plus_h]
rhs = np.log(realized_forward(example_daily_var, example_t, horizon=HORIZON))
assert np.isclose(lhs, rhs), (lhs, rhs)
print(f"alignment OK: y[t+30] = {lhs:.6f}  ==  log(realized_var at t) = {rhs:.6f}")


In [ ]:
window_slice = example_y.loc[:t_plus_h].iloc[-120:]  # zoomed in for visibility; the real context is 1024 long

plt.figure()
plt.plot(window_slice.index, window_slice.values, color="steelblue", label="y = target_30d (what Timestral sees)")
plt.axvline(example_t, color="black", linestyle="--", label="origin t (context ends here)")
plt.scatter([t_plus_h], [lhs], color="tab:red", zorder=5, label="y[t+30] = step-30 target")
plt.legend()
plt.title(f"Context series and the single value forecast 30 steps ahead (origin t = {example_t.date()})")
plt.show()


## 2. Run Timestral-5M and Timestral-209M

Each ticker's forecasts are made in one batched call per model (`contexts` shape `(n_origins, 1024)`), on GPU if available. Forecasts are cached to disk (keyed by ticker, model size, and input variant) so rerunning this cell after the first time is instant.

Main variant (**smoothed**): context = `y` (the series from section 1), last 1024 values ending at each origin; point forecasts at step 30 from the 21 quantiles, exponentiated back to variance units: `median = exp(q50)`, `mean` = trapezoidal integral of `exp(quantiles)` over `forecast.quantile_levels` with flat tails (`src.timestral_utils.quantile_mean`).

Ablation (**daily**, Timestral-209M only): context = `log(daily GK variance)` instead of the 30-day average; each of the 30 forecast days is converted to a mean variance the same way, then averaged -- to show why the input choice in the main variant matters.

**No-symmetric-averaging reference** (Timestral-5M only): `forecast()` by default also forecasts the negated context and averages the reflected result in ("clampsym"). The fine-tuned model in notebook 03 is trained and evaluated through a single forward pass, without that averaging, so the fair zero-shot reference for it is the same model run with `symmetric_averaging=False`: `ts5m_mean_nosym` / `ts5m_median_nosym`.

In [ ]:
def cached_quantiles(key, model, contexts, symmetric_averaging=True):
    path = os.path.join(CACHE_DIR, f"{key}.npz")
    if os.path.exists(path):
        data = np.load(path)
        return data["quantiles"], data["levels"]
    quantiles, levels = forecast_quantiles(model, contexts, horizon=HORIZON, symmetric_averaging=symmetric_averaging)
    os.makedirs(CACHE_DIR, exist_ok=True)
    np.savez(path, quantiles=quantiles, levels=levels)
    return quantiles, levels


model_5m = load_model("5M")
model_209m = load_model("209M")
MODELS = {"5M": model_5m, "209M": model_209m}

In [ ]:
records = []


def add_records(ticker, origins, realized, prefix, quantiles, levels, suffix=""):
    """Step-30 median and mean (variance units) plus the 10-90% band, one row per origin."""
    values = np.exp(quantiles[:, HORIZON - 1, :])  # step 30, variance units
    median = nearest_quantile(values, levels, 0.5)
    mean = quantile_mean(values, levels)
    q10 = nearest_quantile(values, levels, 0.10)
    q90 = nearest_quantile(values, levels, 0.90)
    for model_name, forecast_var in [(f"{prefix}_median{suffix}", median), (f"{prefix}_mean{suffix}", mean)]:
        for i, t in enumerate(origins):
            records.append({
                "ticker": ticker, "origin_date": t, "model": model_name,
                "forecast_var": float(forecast_var[i]), "realized_var": float(realized[i]),
                "q10": float(q10[i]), "q90": float(q90[i]),
            })


for ticker in DEFAULT_TICKERS:
    df = download_ohlc(ticker, start=DEFAULT_START, cache_dir="cache")
    daily_var = garman_klass(df)
    y = target_30d(daily_var, window=30)
    origins = forecast_origins(daily_var.index, TEST_START, horizon=HORIZON, step=STEP)
    realized = np.array([realized_forward(daily_var, t, horizon=HORIZON) for t in origins])

    contexts_smoothed = np.stack([y.loc[:t].iloc[-CONTEXT_LEN:].values for t in origins])
    contexts_daily = np.stack([np.log(daily_var.loc[:t].iloc[-CONTEXT_LEN:].values) for t in origins])
    assert contexts_smoothed.shape == (len(origins), CONTEXT_LEN)

    for size_name, model in MODELS.items():
        quantiles, levels = cached_quantiles(f"{ticker}_{size_name}_smoothed", model, contexts_smoothed)
        add_records(ticker, origins, realized, f"ts{size_name.lower()}", quantiles, levels)

    # no symmetric averaging -- Timestral-5M only, the zero-shot reference for notebook 03
    quantiles, levels = cached_quantiles(f"{ticker}_5M_smoothed_nosym", model_5m, contexts_smoothed, symmetric_averaging=False)
    add_records(ticker, origins, realized, "ts5m", quantiles, levels, suffix="_nosym")

    # daily ablation -- Timestral-209M only
    quantiles_daily, levels = cached_quantiles(f"{ticker}_209M_daily", model_209m, contexts_daily)
    values_daily = np.exp(quantiles_daily)  # (B, H, Q)
    day_means = quantile_mean(values_daily, levels)  # (B, H)
    day_q10 = nearest_quantile(values_daily, levels, 0.10)
    day_q90 = nearest_quantile(values_daily, levels, 0.90)
    daily_mean = day_means.mean(axis=1)
    daily_q10 = day_q10.mean(axis=1)  # average of daily quantiles: NOT a quantile of the 30-day average
    daily_q90 = day_q90.mean(axis=1)

    for i, t in enumerate(origins):
        records.append({
            "ticker": ticker, "origin_date": t, "model": "ts209m_daily_mean",
            "forecast_var": float(daily_mean[i]), "realized_var": float(realized[i]),
            "q10": float(daily_q10[i]), "q90": float(daily_q90[i]),
        })

ts_results = pd.DataFrame(records)

os.makedirs("results", exist_ok=True)
ts_results.to_csv("results/forecasts_timestral_zeroshot.csv", index=False)
if DATA is not None:
    os.makedirs(f"{DATA}/results", exist_ok=True)
    ts_results.to_csv(f"{DATA}/results/forecasts_timestral_zeroshot.csv", index=False)

ts_results.shape

In [ ]:
baselines = pd.read_csv(BASELINES_PATH, parse_dates=["origin_date"])

realized_lookup = (
    baselines.drop_duplicates(["ticker", "origin_date"])
    .set_index(["ticker", "origin_date"])["realized_var"]
)
merged = ts_results.merge(
    realized_lookup.rename("realized_var_baseline"),
    left_on=["ticker", "origin_date"], right_index=True, how="left",
)
assert merged["realized_var_baseline"].notna().all(), "some (ticker, origin_date) pairs are missing from the baseline file -- rerun notebook 01 with the same cache first"
np.testing.assert_allclose(merged["realized_var"].values, merged["realized_var_baseline"].values, rtol=1e-10)
print("realized_var matches results/forecasts_baselines.csv for every (ticker, origin_date).")


SPY example: realized vs HAR (from notebook 01) vs Timestral-209M (mean), with its 10-90% band.


In [ ]:
spy_har = baselines[(baselines.ticker == "SPY") & (baselines.model == "har")].sort_values("origin_date")
spy_ts = ts_results[(ts_results.ticker == "SPY") & (ts_results.model == "ts209m_mean")].sort_values("origin_date")

plt.figure()
plt.plot(spy_har.origin_date, spy_har.realized_var, color="black", alpha=0.6, label="Realized")
plt.plot(spy_har.origin_date, spy_har.forecast_var, label="HAR")
plt.plot(spy_ts.origin_date, spy_ts.forecast_var, label="Timestral-209M (mean)")
plt.fill_between(spy_ts.origin_date, spy_ts.q10, spy_ts.q90, alpha=0.2, label="Timestral 10-90% band")
plt.legend()
plt.title("SPY: realized vs HAR vs Timestral-209M")
plt.show()


## 3. Score table

QLIKE (`src.metrics.qlike`), pooled and per ticker, for GARCH and HAR (loaded from notebook 01's output) alongside every Timestral variant.


In [ ]:
ALL_MODELS = [
    "garch", "har",
    "ts5m_median", "ts5m_mean",
    "ts209m_median", "ts209m_mean", "ts209m_daily_mean",
]

combined = pd.concat(
    [
        baselines[baselines["model"].isin(["garch", "har"])][["ticker", "origin_date", "model", "forecast_var", "realized_var"]],
        ts_results[["ticker", "origin_date", "model", "forecast_var", "realized_var"]],
    ],
    ignore_index=True,
)

print("Pooled QLIKE:")
for model in ALL_MODELS:
    sub = combined[combined["model"] == model]
    print(f"  {model:18s} {qlike(sub['realized_var'].values, sub['forecast_var'].values):.4f}")

rows = []
for ticker in DEFAULT_TICKERS:
    row = {"ticker": ticker}
    for model in ALL_MODELS:
        m = combined[(combined["ticker"] == ticker) & (combined["model"] == model)]
        row[f"qlike_{model}"] = qlike(m["realized_var"].values, m["forecast_var"].values)
    rows.append(row)

score_table = pd.DataFrame(rows).set_index("ticker").round(4)
score_table


## 4. Diebold-Mariano

We compare the best Timestral variant (by pooled QLIKE) against HAR and against GARCH: per ticker (as in notebook 01), and **pooled** by averaging each model's per-observation QLIKE loss across the 10 tickers for each origin date first, then running one DM test on that single averaged loss series (still `lag=5`).


In [ ]:
def qlike_loss(sub):
    r, f = sub["realized_var"].values, sub["forecast_var"].values
    return (r / f) - np.log(r / f) - 1


pooled_qlike = {
    model: qlike(
        combined[combined["model"] == model]["realized_var"].values,
        combined[combined["model"] == model]["forecast_var"].values,
    )
    for model in ["ts5m_median", "ts5m_mean", "ts209m_median", "ts209m_mean", "ts209m_daily_mean"]
}
best_ts = min(pooled_qlike, key=pooled_qlike.get)
print("Best Timestral variant by pooled QLIKE:", best_ts, f"({pooled_qlike[best_ts]:.4f})")


In [ ]:
dm_rows = []
for ticker in DEFAULT_TICKERS:
    m_best = combined[(combined["ticker"] == ticker) & (combined["model"] == best_ts)].sort_values("origin_date")
    m_har = combined[(combined["ticker"] == ticker) & (combined["model"] == "har")].sort_values("origin_date")
    m_garch = combined[(combined["ticker"] == ticker) & (combined["model"] == "garch")].sort_values("origin_date")

    _, _, p_vs_har = dm_test(qlike_loss(m_best), qlike_loss(m_har), lag=5)
    _, _, p_vs_garch = dm_test(qlike_loss(m_best), qlike_loss(m_garch), lag=5)
    dm_rows.append({"ticker": ticker, "dm_pvalue_vs_har": p_vs_har, "dm_pvalue_vs_garch": p_vs_garch})

dm_table = pd.DataFrame(dm_rows).set_index("ticker").round(4)
dm_table


In [ ]:
def per_observation_loss(model):
    sub = combined[combined["model"] == model].copy()
    sub["loss"] = qlike_loss(sub)
    return sub


avg_loss_best = per_observation_loss(best_ts).groupby("origin_date")["loss"].mean().sort_index()
avg_loss_har = per_observation_loss("har").groupby("origin_date")["loss"].mean().sort_index()
avg_loss_garch = per_observation_loss("garch").groupby("origin_date")["loss"].mean().sort_index()

_, _, pooled_p_vs_har = dm_test(avg_loss_best.values, avg_loss_har.values, lag=5)
_, _, pooled_p_vs_garch = dm_test(avg_loss_best.values, avg_loss_garch.values, lag=5)

print(f"Pooled (avg across tickers per origin) DM test, {best_ts} vs HAR:   p = {pooled_p_vs_har:.4f}")
print(f"Pooled (avg across tickers per origin) DM test, {best_ts} vs GARCH: p = {pooled_p_vs_garch:.4f}")


## 5. Calibration

Fraction of `realized_var` falling inside each Timestral variant's 10-90% band -- target 80%. (`_median` and `_mean` rows for the same model share the same underlying quantile forecast and band, so we check each distinct forecast -- the `_mean` rows -- once.)

**Caveat on `ts209m_daily_mean`:** its "q10" / "q90" are the *averages* of the 30 daily 10% / 90% quantiles, not the 10% / 90% quantiles of the 30-day average variance. Averaging quantiles does not give the quantile of the average: day-to-day errors partly cancel in the average, so its true band is much narrower than the averaged daily bands. Its coverage is therefore **not comparable** to the smoothed variants' and says nothing about whether that model is calibrated.

In [ ]:
coverage_rows = []
for model in ["ts5m_mean", "ts5m_mean_nosym", "ts209m_mean", "ts209m_daily_mean"]:
    sub = ts_results[ts_results["model"] == model]
    inside = (sub["realized_var"] >= sub["q10"]) & (sub["realized_var"] <= sub["q90"])
    coverage_rows.append({"model": model, "coverage": inside.mean()})

coverage_table = pd.DataFrame(coverage_rows)
coverage_table

## 6. Leakage note

Timestral's GitHub repository (`mistralai/timestral`) has a single commit, *"Timestral release"*, dated **2026-10-15**. There is no earlier commit history, and the Hugging Face model card does not state a training-data cutoff.

Our test period runs from 2022-01-01 to whatever the last fully-realized origin is when this is run -- around 2026-09, a few weeks *before* Timestral's release. That means **every forecast origin in this evaluation predates Timestral's public release**: there is currently no post-release window in our test period to use as a clean, provably-unseen holdout. If Timestral's pretraining corpus includes financial time series covering this window -- which we have no way to confirm or rule out from the public model card -- the zero-shot numbers above could be flattered by memorization rather than genuine generalization. This is a real limitation of this evaluation, not a hypothetical one.

The cell below checks anyway, so this becomes informative automatically once this notebook is rerun later with more data:


In [ ]:
RELEASE_DATE = pd.Timestamp("2026-10-15")
post_release = combined[combined["origin_date"] > RELEASE_DATE]

if len(post_release) > 0:
    print(f"{len(post_release)} observations after the release date. Pooled QLIKE on that subset:")
    for model in ALL_MODELS:
        sub = post_release[post_release["model"] == model]
        if len(sub) > 0:
            print(f"  {model:18s} {qlike(sub['realized_var'].values, sub['forecast_var'].values):.4f}")
else:
    print(
        "No forecast origins fall after Timestral's release date (2026-10-15) yet -- "
        "see the note above: this evaluation cannot currently rule out leakage."
    )


## 7. Full comparison from the saved files

Re-reads `forecasts_baselines.csv` (notebook 01) and `forecasts_timestral_zeroshot.csv` (this notebook) from disk and merges them on `(ticker, origin_date)`, asserting `realized_var` is identical in both, then scores every model on the merged set (`src.evaluation`):

- QLIKE pooled, per ticker, and per year (the year of the forecast origin);
- Diebold-Mariano on QLIKE losses (`lag=5`) vs HAR and vs GARCH, per ticker and pooled (average loss across the 10 tickers per origin date, then one DM test). `mean_diff < 0` means the Timestral model has the lower loss.

In [ ]:
from src.evaluation import calibration_table, dm_table, load_forecasts, qlike_tables

ZS_PATH = f"{DATA}/results/forecasts_timestral_zeroshot.csv" if DATA else "results/forecasts_timestral_zeroshot.csv"
full = load_forecasts(BASELINES_PATH, ZS_PATH)  # asserts realized_var matches on every (ticker, origin_date)
print("merged:", full.shape, "| realized_var identical in both files")

COMPARE = ["garch", "har", "ts5m_mean", "ts5m_mean_nosym", "ts209m_mean", "ts209m_daily_mean"]
pooled, per_ticker, per_year = qlike_tables(full, COMPARE)
pooled.round(4).to_frame().T

In [ ]:
per_ticker.round(4)

In [ ]:
per_year.round(4)

In [ ]:
TS_MODELS = ["ts5m_mean", "ts5m_mean_nosym", "ts209m_mean", "ts209m_daily_mean"]
dm_all = {(m, ref): dm_table(full, m, ref, lag=5) for m in TS_MODELS for ref in ["har", "garch"]}

dm_pvalues = pd.DataFrame({f"{m} vs {ref}": t["p_value"] for (m, ref), t in dm_all.items()})
dm_pvalues.round(4)

In [ ]:
pd.DataFrame({f"{m} vs {ref}": t.loc["pooled"] for (m, ref), t in dm_all.items()}).T.round(4)

In [ ]:
calibration_table(full, COMPARE).round(4)

## 8. Conclusion

Fill this in after actually running the notebook -- the numbers below are placeholders, not results:

- **Pooled QLIKE ranking**: garch = ___, har = ___, best Timestral variant (`___`) = ___. Does it **beat / match / lose to** HAR? (beat = lower QLIKE *and* a significant pooled DM p-value in section 4; match = similar QLIKE, DM not significant; lose = higher QLIKE, or DM significant in HAR's favor.)
- **Calibration** (section 5): coverage was ___% against an 80% target -- overcovered (band too wide / underconfident) or undercovered (band too narrow / overconfident)?
- **Daily ablation**: how much worse, or how much less stable, was `ts209m_daily_mean` than the smoothed `ts209m_mean`? That gap is the evidence for *why* we feed the 30-day average rather than raw daily variance.
- **Leakage** (section 6): given that every origin predates Timestral's release, treat any "Timestral beats HAR" result here as provisional, not proof of forecasting skill, until it can be rerun on genuinely post-release data.

**What fine-tuning would need to fix** depends on which of these is the weak point:
- Forecasts **systematically too high or too low** vs. realized_var (check the SPY plot in section 2) -> a bias fine-tuning should fix directly.
- QLIKE is fine but the **band is miscalibrated** (section 5) -> the quantile head specifically needs fine-tuning, not just the median.
- Tracks the general level but **reacts too slowly after volatility shocks** (visible as lag around sharp realized spikes in the SPY plot) -> a recency/context-weighting issue that fine-tuning on this exact target and domain may or may not resolve.